<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/Data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade pip
!pip install torch

In [2]:
!pip install scikit-learn

In [3]:
!pip install Pillow
!pip install pandas
!pip install matplotlib

In [4]:
import json
import os
import torch
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image
import pandas as pd
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
pd.set_option('display.max_columns', None)

In [6]:
# unzip the VCell simulation images
!tar -xf "data/all_images.zip" -C '/Users/reeshapatel/software1/temp'
#!tar -xf "/Users/mikhailblinov/Software/PatternsFormation/Toybox/05-09-2024_all_images.zip" -C '/Users/mikhailblinov/Software/Temp'
#!tar -xf "/Users/mikhailblinov/Software/PatternsFormation/Toybox/Generated Images.zip" -C '/Users/mikhailblinov/Software/Temp'
#!tar -xf "/Users/mikhailblinov/Software/PatternsFormation/Toybox/Generated_Images_7-2-24.zip" -C '/Users/mikhailblinov/Software/Temp'
!echo Unzip Complete

Unzip Complete


In [7]:
# downgrade numpy to open pickle files
# Numpy: 2.0.2
# Pickle Highest Protocol: 5 (no need to change)
# Pickle Default Protocol: 4 (no need to change)

!pip uninstall --yes numpy
!pip install numpy==2.0.2

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
  Using cached numpy-2.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (114 kB)
Using cached numpy-2.0.2-cp312-cp312-macosx_11_0_arm64.whl (13.5 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
molclustpy 0.1.0 requires bionetgen, which is not installed.


In [3]:
# import features dataframe for all the images
with open("data/unclassified_features.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

print(feats_df.shape)

feats_df

(39517, 111)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.011652,0.1066,0.115805,0.061599,-0.043244,0.007928,0.559698,0,2,0.png,0,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
1,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3/,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3.136500e+03,649.910626,100.0,0.0,3.136500e+03,649.910626,97.942857,58.572760,96.014286,58.843981,120.129000,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
2,0.014455,0.085124,0.091124,0.113974,-0.060333,0.001776,1.798215,0,2,2.png,2,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
3,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images

In [29]:
# compare all pkl files
file1 = "data/manual_classification_cl6.pkl"
# file1 originally named 2024-08-15_manual_classification.pkl

file2 = "data/2024-08-18_curr_df.pkl"
# file2 - CNN classification

file3 = "data/2024-08-19_feat_array_unscaled.pkl"

file4 = "data/2024-08-19_feats_df_narrow.pkl"
# file4 - unclassified_features shortened

file5 = "data/2025-05-24_feats_df_toybox.pkl"

file6 = "data/Images_Classified.pkl"
# file6 - 27515 images and 113 columns - fullpath, classifier_pred_class

file7 = "data/manual_classification.pkl"

file8 = "data/single_param_scans.pkl"

file9 = "data/unclassified_features.pkl"

file10 = "data/random_indices_array.pkl"
# file10 - originally named "2024-08-11_random_indices_array.pkl"

file11 = "../OlderStuff/2024-08-11_manual_classification.pkl"

In [43]:
with open(file11, 'rb') as f:
  df = pickle.load(f)

print(df.shape)
df

df_none = df[df['class'] == 'None']
df_none
print(df_none.shape)

(39517, 5)
(13241, 5)


In [26]:
# function to compare data frames and dictionaries
def compare_data_structures(ds1, ds2, f1, f2):
    if isinstance(ds1, pd.DataFrame) and isinstance(ds2, pd.DataFrame):
        print("Both are DataFrames")
        comparison = ds1.equals(ds2)
        print(f"DataFrames are equal: {comparison}")
    elif isinstance(ds1, dict) and isinstance(ds2, dict):
        print("Both are Dictionaries")
        keys1 = set(ds1.keys())
        keys2 = set(ds2.keys())
        common_keys = keys1.intersection(keys2)
        for key in common_keys:
            if ds1[key] != ds2[key]:
                print(f"Difference in key '{key}': {ds1[key]} != {ds2[key]}")
    else:
        print(f1, "is type", type(ds1))
        print(f2, "is type", type(ds2))


def find_identical_rows(df1, df2, key_columns=None):
    identical_rows = pd.merge(df1, df2, how='inner', on=key_columns) # key_columns is a list of columns to match on
    return identical_rows


# Example usage:
with open("data/2024-08-15_curr_df.pkl", 'rb') as f1, open(file2, 'rb') as f2:
    ds1 = pickle.load(f1)
    ds2 = pickle.load(f2)
    compare_data_structures(ds1, ds2, f1, f2)

ds1.head()
ds2.head()

Both are DataFrames
DataFrames are equal: False


,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348
